# RMSX + Flipbook Molstar Colab Demo

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AntunesLab/rmsx/blob/rmsx_molstar/RMSX_Molstar_Colab_Demo.ipynb)

This notebook is the Google Colab-friendly RMSX/Flipbook demo. It keeps the main Quickstart flow, but uses **Molstar** for every 3D Flipbook visualization so you can inspect protein motion directly in the notebook without installing ChimeraX or VMD locally.

We'll cover:

1. Environment setup
2. Loading bundled demo input files
3. Single-chain RMSX
4. Multi-chain RMSX
5. Interactive Molstar Flipbooks
6. Masked Molstar Flipbooks

> In Colab, choose **Runtime > Run all**. The first setup cells may take a few minutes because RMSX, MDAnalysis, and R plotting packages are installed into the temporary Colab runtime.


## 1) Environment Setup

RMSX uses Python/MDAnalysis for trajectory analysis, R for heatmaps and RMSD/RMSF plots, and Molstar for the inline 3D viewer. This notebook installs RMSX from GitHub when it detects that it is running in Colab.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
GITHUB_REF = os.environ.get("RMSX_GITHUB_REF", "rmsx_molstar")

if IN_COLAB:
    print(f"Installing RMSX from AntunesLab/rmsx@{GITHUB_REF} ...")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        f"git+https://github.com/AntunesLab/rmsx.git@{GITHUB_REF}",
    ])
else:
    print("Not running in Colab; using the RMSX package available in this environment.")

print("Python:", sys.version.split()[0])
print("Running in Colab:", IN_COLAB)


### R Plotting Packages

The Molstar viewer itself does not need R, but the RMSX demo keeps the original heatmap/RMSD/RMSF outputs. If `Rscript` is missing in Colab, this cell installs R first, then installs the R packages used by RMSX plotting.


In [ ]:
RSCRIPT = os.environ.get("RSCRIPT", "Rscript")

if shutil.which(RSCRIPT) is None and IN_COLAB:
    print("Rscript not found; installing R for Colab plotting support...")
    subprocess.check_call(["apt-get", "update", "-qq"])
    subprocess.check_call(["apt-get", "install", "-y", "-qq", "r-base"])

if shutil.which(RSCRIPT) is None:
    print("Rscript not found. RMSX analysis can run, but plot generation may be skipped or fail until R is available.")
else:
    r_setup = r'''
user <- Sys.getenv("R_LIBS_USER")
if (user == "") user <- file.path(Sys.getenv("HOME"), "R", "library")
dir.create(user, recursive = TRUE, showWarnings = FALSE)
.libPaths(c(user, .libPaths()))
pkgs <- c("ggplot2", "viridis", "dplyr", "tidyr", "stringr", "readr", "gridExtra")
missing <- pkgs[!vapply(pkgs, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing)) {
  install.packages(missing, lib = user, repos = "https://cloud.r-project.org", dependencies = c("Depends", "Imports", "LinkingTo"))
} else {
  cat("R packages already installed\n")
}
'''
    subprocess.check_call([RSCRIPT, "-e", r_setup])
    print("Rscript ready:", shutil.which(RSCRIPT))


## 2) Load Demo Input Files

The notebook looks for the bundled demo inputs inside the installed `rmsx` package first. If you are running from a source checkout, it also checks local `test_files` folders. Demo outputs are written to `./rmsx_demo_outputs` so the packaged inputs remain read-only.


In [ ]:
from IPython.display import HTML, Image, display
import glob
import json

import rmsx
from rmsx import all_chain_rmsx, run_rmsx, run_rmsx_flipbook, write_molstar_flipbook

pkg_dir = Path(rmsx.__file__).resolve().parent
env_override = os.environ.get("RMSX_TEST_DIR")

candidates = []
if env_override:
    candidates.append(Path(env_override))

candidates += [
    pkg_dir / "test_files",
    Path.cwd() / "test_files",
    Path.cwd() / "rmsx" / "test_files",
    pkg_dir.parent / "test_files",
]

test_dir = next((p for p in candidates if p.exists()), None)

if not test_dir:
    repo_dir = Path.cwd() / "rmsx"
    if not repo_dir.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", "--branch", GITHUB_REF, "https://github.com/AntunesLab/rmsx.git", str(repo_dir)])
    test_dir = repo_dir / "rmsx" / "test_files"
    if not test_dir.exists():
        test_dir = repo_dir / "test_files"
    if not test_dir.exists():
        raise FileNotFoundError("Could not locate bundled RMSX demo inputs.")

demo_output_root = Path.cwd() / "rmsx_demo_outputs"
demo_output_root.mkdir(exist_ok=True)

pdb_file = (test_dir / "1UBQ.pdb").as_posix()
dcd_file = (test_dir / "mon_sys.dcd").as_posix()
output_dir = (demo_output_root / "example_uqb").as_posix()

pdb_file_multi = (test_dir / "protease_backbone.pdb").as_posix()
traj_file_multi = (test_dir / "short_protease_backbone.dcd").as_posix()
output_dir_multi = (demo_output_root / "protease").as_posix()

print("RMSX package:", Path(rmsx.__file__).resolve())
print("Demo input directory:", test_dir)
print("Demo output root:", demo_output_root.resolve())
print("Single-chain input:", pdb_file, dcd_file, sep="
  ")
print("Multi-chain input:", pdb_file_multi, traj_file_multi, sep="
  ")


## 3) Single-Chain RMSX

This is the original Ubiquitin-style demo path. `run_rmsx` computes per-slice RMSX values and writes heatmaps, RMSD/RMSF plots, RMSX tables, and slice PDB files. For the bundled Ubiquitin example, the chain ID is `"7"`.


In [ ]:
run_rmsx(
    topology_file=pdb_file,
    trajectory_file=dcd_file,
    output_dir=output_dir,
    num_slices=9,
    slice_size=None,
    rscript_executable=os.environ.get("RSCRIPT", "Rscript"),
    verbose=False,
    interpolate=False,
    triple=True,
    overwrite=True,
    palette="mako",
    chain_sele="7",
    start_frame=0,
    end_frame=None,
    full_backbone=True,
)

print("Single-chain RMSX output:", output_dir)


In [ ]:
single_images = sorted(glob.glob(os.path.join(output_dir, "*.png")))
if single_images:
    display(Image(filename=single_images[-1]))
else:
    print("No PNG outputs found yet:", output_dir)


## 4) Multi-Chain RMSX

The protease demo runs RMSX on each chain and synchronizes the color scale so chain-level plots and the combined Flipbook use a consistent RMSX range.


In [ ]:
combined_dir = all_chain_rmsx(
    topology_file=pdb_file_multi,
    trajectory_file=traj_file_multi,
    output_dir=output_dir_multi,
    num_slices=9,
    slice_size=None,
    rscript_executable=os.environ.get("RSCRIPT", "Rscript"),
    verbose=False,
    interpolate=False,
    triple=True,
    overwrite=True,
    palette="turbo",
    start_frame=0,
    end_frame=None,
    sync_color_scale=True,
    full_backbone=True,
)

combined_dir = Path(combined_dir)
print("Protease combined output:", combined_dir)
print("Slice PDB count:", len(list(combined_dir.glob("slice_*_first_frame.pdb"))))


## 5) Interactive Molstar Flipbook

This is the Colab-friendly Flipbook path. `viewer="molstar"` writes an interactive HTML viewer and displays it directly in the notebook output. The same folder can still be downloaded later if you want the generated PDB slices or standalone HTML.


In [ ]:
protease_molstar = write_molstar_flipbook(
    combined_dir,
    palette="turbo",
    spacing_factor=0.44,
    camera_mode="orthographic",
    output_html=demo_output_root / "protease_molstar_flipbook.html",
    output_manifest=demo_output_root / "protease_molstar_manifest.json",
    asset_mode="cdn",
    iframe_height=720,
)

display(protease_molstar)
print("Standalone HTML:", protease_molstar.html_path)
print("Manifest:", protease_molstar.manifest_path)


### Full Analysis + Molstar in One Call

If you want RMSX and the Molstar Flipbook in one step for your own trajectory, use `run_rmsx_flipbook(..., viewer="molstar")`. This reruns the protease example so the complete trajectory-to-viewer path is visible in one cell.


In [ ]:
one_step_output = demo_output_root / "protease_one_step_molstar"

one_step_molstar = run_rmsx_flipbook(
    topology_file=pdb_file_multi,
    trajectory_file=traj_file_multi,
    output_dir=one_step_output,
    num_slices=9,
    slice_size=None,
    rscript_executable=os.environ.get("RSCRIPT", "Rscript"),
    verbose=False,
    interpolate=False,
    triple=True,
    overwrite=True,
    palette="magma",
    spacingFactor="0.44",
    viewer="molstar",
    molstar_asset_mode="cdn",
    molstar_height=720,
    molstar_camera_mode="orthographic",
    start_frame=0,
    end_frame=None,
)

display(one_step_molstar)
print("One-step Molstar output:", one_step_output)


## 6) Masked Molstar Flipbook

Masking is useful when a very mobile or disordered region dominates the RMSX scale. Masked residues are excluded from the range estimate, shown with hatch overlays in heatmaps, and rendered semi-transparent in Molstar.


In [ ]:
protease_active_site_mask = [
    "segid A and resid 45:55",
    "segid B and resid 45:55",
]

masked_output = demo_output_root / "protease_masked_molstar"

masked_molstar = run_rmsx_flipbook(
    topology_file=pdb_file_multi,
    trajectory_file=traj_file_multi,
    output_dir=masked_output,
    num_slices=9,
    rscript_executable=os.environ.get("RSCRIPT", "Rscript"),
    verbose=False,
    interpolate=False,
    triple=True,
    overwrite=True,
    palette="magma",
    spacingFactor="0.44",
    sync_color_scale=True,
    mask=protease_active_site_mask,
    viewer="molstar",
    molstar_asset_mode="cdn",
    molstar_height=720,
    molstar_camera_mode="orthographic",
)

display(masked_molstar)
print("Masked Molstar output:", masked_output)


## Use Your Own Data

Replace `topology_file`, `trajectory_file`, and `output_dir` with your own files. For Colab, upload files through the left sidebar or mount Google Drive first.

```python
from rmsx import run_rmsx_flipbook

result = run_rmsx_flipbook(
    topology_file="/content/my_structure.pdb",
    trajectory_file="/content/my_trajectory.dcd",
    output_dir="/content/my_rmsx_flipbook",
    num_slices=9,
    chain_sele=None,
    palette="turbo",
    viewer="molstar",
    molstar_asset_mode="cdn",
    molstar_height=720,
    molstar_camera_mode="orthographic",
    spacingFactor="0.44",
    overwrite=True,
)

display(result)
```


## Citation

If you use RMSX + Flipbook in your work, please cite:

Beruldsen, F., de Freitas, M.V. & Antunes, D.A. *High resolution mapping of protein motions in time and space with RMSX and Flipbook.* **Scientific Reports** (2026). https://doi.org/10.1038/s41598-026-39869-7
